# The regression

**Step 2 (this part):** decide exactly what the model measures, and write it down **before**
fitting it. **Step 3 (added later):** fit it and read the results.

Why write the rules first? If you fit a model, look at the result, then adjust it, it's easy to
keep adjusting until it says what you hoped. Fixing the choices in advance, including which
checks will be run and what result would count as support, removes that temptation. Every check
listed below gets reported in step 3, whatever it shows.

The reasoning behind each choice comes from `01_explore.ipynb`.

## The question

> Holding time of day, airline, flight type and wind constant, are flights into **SFO** more
> likely to arrive late than flights into **ATL, EWR, LAX and MIA**?

## The decisions

| # | Decision | Choice | Why | Rejected alternative |
|---|---|---|---|---|
| 1 | Outcome | **Late yes/no** (arrived 15+ min after schedule), so **logistic regression** | It's the standard airline and government definition of punctuality, so the result is easy to explain and compare. Delay *minutes* are dominated by early arrivals (median −12 min) and capped at 56 min, so a minutes model would mostly describe how early flights land | Linear regression on delay minutes |
| 2 | Pulls included | The **11 regular pulls**, 7 Sep – 1 Oct | Consistent collection, and every flight has weather. The 4–5 Sep manual pulls held nearly all daytime flights and the flights without weather | All pulls (check R6) |
| 3 | Flights included | Only flights **scheduled at least 2 hours before their pull** | A late flight scheduled shortly before the 09:00 UTC pull is still in the air, so the API never returns it: flights in that window show 1–5% late vs about 14% otherwise. LAX and MIA have the most such flights, so keeping them would make those airports look better and inflate SFO's gap. The schedule is set in advance, so this filter can't be influenced by how the airport performs | Keep them all (check R7) |
| 4 | Operators included | Scheduled airlines and cargo. **Private jets and air ambulances excluded** | They fly on demand, not to a published schedule, so "late" doesn't mean the same thing | — |
| 5 | Airport | Compared as **SFO vs the other four together** (primary), then **each airport vs SFO** (secondary) | One primary test answers the main question without the false-alarm risk of four separate tests | Only pairwise tests |
| 6 | Time of day | **Night** (scheduled 00:00–05:59) vs the rest | On the regular pulls there is essentially no morning and little afternoon, so finer slots can't be estimated | 4 slots |
| 7 | Airline | United (baseline), Delta, American, Alaska: the airlines with **75+ flights** in this sample, **fixed** for every check and for October. The rest become Other | **Alaska must be separate**: 55% of its flights in this sample go to SFO, so lumping it into Other would blur the SFO comparison. The cut-off of 75 flights was chosen *to keep Alaska* (it has 93). Check R8 tests a lower cut-off, so the choice of cut-off doesn't decide the answer | A 100-flight cut-off, which would drop Alaska |
| 8 | Flight type | **Cargo** yes/no | Cargo is rarely late and concentrated at MIA and LAX | — |
| 9 | Weather | **Wind speed** (mph) | The only weather column that varies at more than one airport and can be trusted (step 1, section 3) | Rain, visibility, the weather label |

**What "holding wind constant" means here.** SFO is far windier than the others (about 16 mph on
average vs 4 at ATL). With wind in the model, the SFO result is **SFO's effect beyond what its
wind explains**, closer to an operational effect. Check R1 drops wind to show the **total** SFO
effect. Both are reported.

**Deliberately left out:**
- **Departure delay.** It's one of the ways an airport makes flights late, so controlling for it
  would hide the effect being measured.
- **Rain, visibility, the weather label.** Step 1 showed each is either unreliable or confined to one airport.
- **Flight length.** Not expected to matter much (step 1 review), so it's a check (R2) rather than in the main model.

## How uncertainty is measured

Flights in the same API response (one pull at one airport, typically 25–50 flights after
codeshares are merged) share that evening's conditions: a storm, an air-traffic hold. They aren't
independent, so standard errors are **clustered by batch** (pull × airport).

That gives only 11 batches per airport, so any SFO comparison rests on about 22 batches. With that
few, clustered confidence intervals tend to come out **too narrow**, which leans toward false
positives. So every claim must also pass a **cluster bootstrap**: resample each airport's 11
batches with replacement, re-fit the model 999 times, and take the middle 95% of the results as
the confidence interval. Some resamples may fail to fit (night has only 12 late flights). Those
are skipped and counted; if more than 5% fail, the bootstrap is reported as unreliable.

## Checks, fixed in advance (all reported in step 3)

| Check | What changes | What it tests |
|---|---|---|
| R1 | Drop wind speed | The **total** SFO effect, including what wind explains |
| R2 | Add scheduled flight length | Is SFO's result really a long-haul effect? |
| R3 | Passenger flights only | Is the result driven by cargo? |
| R4 | Add an indicator for each pull | Do pull-wide conditions explain it? |
| R5 | Re-fit 11 times, leaving out one pull each time | Does any single evening drive the result? |
| R6 | Include the 4–5 Sep pulls (without wind, so compare with R1). The 2-hour filter isn't applied to those flights, since those pulls didn't run at 09:00. Clusters: departure date × airport | Does the regular-pull restriction change the answer? |
| R7 | Keep the flights scheduled within 2 hours of the pull | How much did the capture bias matter? |
| R8 | Airline cut-off of 40 flights instead of 75 | Does the cut-off choice matter? |

## What will count as a result

**Primary test:** SFO vs the other four airports together. **"SFO is later"** if its odds ratio is
above 1 and **both** 95% intervals (clustered and bootstrap) stay above 1.

**Secondary tests**, each airport vs SFO, with p-values **Holm-adjusted** for running four tests:
- **"SFO is later than X"**: X's odds ratio is below 1 with adjusted p < 0.05 and a bootstrap interval below 1.
- **"X is later than SFO"**: the mirror image.
- Otherwise **"not distinguishable from SFO"**.

**"Robust"** means the claim's interval still excludes 1 in R1–R4, and its direction holds in
every leave-one-out fit (R5) and in R6–R8.

**Expectation, written now so it can't be adjusted later:** from step 1, SFO should be later than
the others overall, clearly later than ATL and EWR, borderline against LAX, and not
distinguishable from MIA. Fixing the capture bias may move LAX and MIA, since both were
flattered by it. Whatever the model says gets reported as it is.

## An honest limit of this pre-registration

These hypotheses came from the same September data the model will use. Writing the rules down
first stops the model from being tuned toward an answer, but it doesn't make this a true test. **The
true test is October:** once about 10 new pulls have landed (around 21 October), step 3's code is
re-run unchanged on **pulls from 3 October onward** (the 1 October pull is already in this sample),
data that played no part in forming these hypotheses.

The **primary test** decides. If SFO is later there too, the result is confirmed. If not, it is
**not confirmed**. That could mean the September result was noise, or simply that one month of
data is too small to detect a real but modest effect. The two can't be told apart without more data.

## Building the model table

Everything below checks that the data can support the model as specified. Nothing is fitted yet.

In [1]:
import pandas as pd
from patsy import dmatrix
from statsmodels.stats.outliers_influence import variance_inflation_factor

from flights_data import load_flights, prepare

pd.set_option("display.precision", 3)

flights = load_flights()
clean = prepare(flights)

Loaded live data from Snowflake
2,722 flights


In [2]:
FIRST_REGULAR_PULL = "2026-09-07"
MIN_HOURS_BEFORE_PULL = 2

# Airlines with 75+ flights in this sample, fixed on 2 October 2026. The list stays the same in
# every check and in the October re-run, so a check never changes two things at once
# (only check R8 changes it deliberately).
MODEL_CARRIERS = ["United Airlines", "Delta Air Lines", "American Airlines", "Alaska Airlines"]

sample = clean[
    (clean["pull_date"] >= FIRST_REGULAR_PULL)
    & (clean["hours_before_pull"] >= MIN_HOURS_BEFORE_PULL)
    & ~clean["is_non_scheduled"]
].copy()

sample["is_late"] = sample["is_delayed_arrival"].astype(int)
sample["is_night"] = (sample["arrival_hour"] < 6).astype(int)
sample["carrier_group"] = sample["carrier"].where(sample["carrier"].isin(MODEL_CARRIERS), "Other")

# The evidence for the fixed list: flights per airline in this sample.
print(sample["carrier"].value_counts().head(6))
sample["cargo"] = sample["is_cargo"].astype(int)
sample["is_sfo"] = sample["is_sfo"].astype(int)

# One batch = one API response: one pull at one airport. Flights in a batch share conditions.
sample["batch"] = sample["pull_date"].dt.strftime("%Y-%m-%d") + " " + sample["arrival_airport"]

carrier
United Airlines      484
Delta Air Lines      339
American Airlines    298
Alaska Airlines       93
Air Canada            72
Frontier              70
Name: count, dtype: int64


### The model, written out

These are the exact formulas step 3 will fit. The two differ only in how airport enters: the
primary model has a single SFO yes/no input; the secondary model has one input per airport,
with `Treatment("SFO")` making SFO the baseline. `Treatment("United Airlines")` makes United
the baseline airline in both.

In [3]:
CONTROLS = (
    'is_night'
    ' + C(carrier_group, Treatment("United Airlines"))'
    ' + cargo'
    ' + weather_wind_speed'
)
PRIMARY_FORMULA = "is_late ~ is_sfo + " + CONTROLS
SECONDARY_FORMULA = 'is_late ~ C(arrival_airport, Treatment("SFO")) + ' + CONTROLS
INPUTS = SECONDARY_FORMULA.split("~")[1]   # used by the checks below

print("Primary:  ", PRIMARY_FORMULA)
print("Secondary:", SECONDARY_FORMULA)

Primary:   is_late ~ is_sfo + is_night + C(carrier_group, Treatment("United Airlines")) + cargo + weather_wind_speed
Secondary: is_late ~ C(arrival_airport, Treatment("SFO")) + is_night + C(carrier_group, Treatment("United Airlines")) + cargo + weather_wind_speed


### Check 1: is there enough data?

In [4]:
number_of_inputs = dmatrix(INPUTS, sample).shape[1] - 1   # minus the intercept

pd.Series({
    "flights": len(sample),
    "late flights": sample["is_late"].sum(),
    "pulls": sample["pull_date"].nunique(),
    "batches (pull x airport)": sample["batch"].nunique(),
    "model inputs": number_of_inputs,
    "late flights per input": round(sample["is_late"].sum() / number_of_inputs, 1),
})

flights                     1726.0
late flights                 250.0
pulls                         11.0
batches (pull x airport)      55.0
model inputs                  11.0
late flights per input        22.7
dtype: float64

A common rule of thumb asks for at least 10 late flights per input. More is better.

### Check 2: does every group have flights and late flights?

A group with no late flights at all would break a logistic regression.

In [5]:
for column in ["arrival_airport", "carrier_group", "is_night", "cargo"]:
    print(sample.groupby(column)["is_late"].agg(flights="count", late="sum").T, "\n")

arrival_airport  ATL  EWR  LAX  MIA  SFO
flights          305  465  287  399  270
late              35   56   40   64   55 

carrier_group  Alaska Airlines  American Airlines  Delta Air Lines  Other  \
flights                     93                298              339    512   
late                        21                 70               38     58   

carrier_group  United Airlines  
flights                    484  
late                        63   

is_night     0    1
flights   1589  137
late       238   12 

cargo       0    1
flights  1557  169
late      236   14 



### Check 3: batches per airport

How many batches each airport's result rests on, and how big they are.

In [6]:
batch_sizes = sample.groupby(["arrival_airport", "batch"]).size()
batch_sizes.groupby("arrival_airport").agg(batches="count", smallest="min", largest="max")

,batches,smallest,largest
arrival_airport,,,
ATL,11,22,30
EWR,11,39,45
LAX,11,22,29
MIA,11,30,43
SFO,11,21,26


### Check 4: no missing values in the model columns

In [7]:
MODEL_COLUMNS = ["is_late", "arrival_airport", "is_night", "carrier_group", "cargo", "weather_wind_speed"]
sample[MODEL_COLUMNS].isna().sum()

is_late               0
arrival_airport       0
is_night              0
carrier_group         0
cargo                 0
weather_wind_speed    0
dtype: int64

### Check 5: are any inputs too entangled to separate?

The variance inflation factor (VIF) measures how much an input overlaps with the others. Around
1 means independent. Above about 5 means the model will struggle to tell that input's effect
apart from the others. This uses the secondary model, which has more inputs to overlap.

In [8]:
design = dmatrix(INPUTS, sample, return_type="dataframe")

vif = pd.Series(
    [variance_inflation_factor(design.values, i) for i in range(design.shape[1])],
    index=design.columns,
).drop("Intercept")

vif.round(2).sort_values(ascending=False)

C(arrival_airport, Treatment("SFO"))[T.MIA]                            3.66
C(arrival_airport, Treatment("SFO"))[T.ATL]                            3.43
C(arrival_airport, Treatment("SFO"))[T.EWR]                            2.79
C(arrival_airport, Treatment("SFO"))[T.LAX]                            2.44
C(carrier_group, Treatment("United Airlines"))[T.Other]                2.09
weather_wind_speed                                                     2.03
C(carrier_group, Treatment("United Airlines"))[T.Delta Air Lines]      1.64
C(carrier_group, Treatment("United Airlines"))[T.American Airlines]    1.60
cargo                                                                  1.44
C(carrier_group, Treatment("United Airlines"))[T.Alaska Airlines]      1.28
is_night                                                               1.17
dtype: float64

### Check 6: how much overlap is there between airline and airport?

Share of each airline group's flights going to each airport.

In [9]:
pd.crosstab(sample["carrier_group"], sample["arrival_airport"], normalize="index").round(2)

arrival_airport,ATL,EWR,LAX,MIA,SFO
carrier_group,,,,,
Alaska Airlines,0.10,0.00,0.35,0.00,0.55
American Airlines,0.11,0.16,0.13,0.46,0.13
Delta Air Lines,0.47,0.16,0.09,0.15,0.13
Other,0.10,0.17,0.24,0.38,0.12
United Airlines,0.11,0.57,0.12,0.04,0.15


## Step 2 conclusion

_Snapshot from 2 October 2026. The checks above update on re-run; this text does not._

**The data can support the model as specified.**

- **Size:** 1,726 flights and 250 late, across 11 pulls, after the three filters (regular pulls,
  scheduled 2+ hours before the pull, scheduled operators). With 11 inputs that's about 23 late
  flights per input, comfortably above the rule of thumb of 10.
- **Every group has late flights**, so nothing will break the fit. The thinnest are night
  (12 late), cargo (14) and Alaska (21). Their own estimates will be imprecise, but they're
  controls, not the question.
- **No missing values** in any model column.
- **Overlap between inputs is acceptable:** every VIF is under 4. The airport inputs sit around
  3 because each is compared with SFO, the smallest airport group (270 flights).
- **Airline and airport overlap but can still be separated:** Alaska sends 55% of its flights to
  SFO and United 57% to EWR, yet every airline flies to at least three of the five airports.
- **Each airport rests on 11 batches of 21–45 flights.** That's the honest size of the evidence,
  and it's why every claim must also pass the cluster bootstrap.

**The specification above is now fixed.** Step 3 fits `PRIMARY_FORMULA` and `SECONDARY_FORMULA`
exactly as printed, runs the cluster bootstrap and checks R1 to R8, and reports all of them against
the rules in "What will count as a result". Then, around 21 October, the same code runs on pulls
from 3 October onward, as the true test.